In [1]:
import sqlite3
import requests
import time
import json
import pandas as pd
import urllib.parse

In [2]:
DB_PATH = 'gaming_warehouse.db'

In [3]:
#Crear la tabla de hsitorico de reviews
def preparar_tabla_reviews():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS Hist_Steam_Reviews (
            resena_id TEXT PRIMARY KEY,
            juego_id INTEGER,
            resena_texto TEXT,
            recomendado INTEGER, -- (voted_up)
            votos_utiles INTEGER, -- (votes_up)
            votos_graciosos INTEGER, -- (votes_funny)
            puntuacion_ponderada REAL, -- (weighted_vote_score)
            minutos_al_resenar INTEGER, -- (playtime_at_review)
            minutos_totales INTEGER, -- (playtime_forever)
            fecha_creacion_unix INTEGER, -- (timestamp_created)
            autor_num_resenas INTEGER,
            autor_num_juegos INTEGER,
            recibido_gratis INTEGER,
            escrito_acceso_anticipado INTEGER,
            FOREIGN KEY(juego_id) REFERENCES CAT_Juego(juego_id)
        );
    """)
    conn.commit()
    conn.close()


In [4]:
preparar_tabla_reviews()

In [ ]:
def descargar_reviews_steam(max_reviews_por_juego=100, min_reviews_juego=20, limite_juegos=10):
    conn_principal = sqlite3.connect(DB_PATH)
    
    query = """
        SELECT 
            j.juego_id, 
            j.id_steam, 
            j.titulo, 
            j.recommendations_count,
            COALESCE(r.reviews_en_bd, 0) as reviews_en_bd
        FROM CAT_Juego j
        LEFT JOIN (
            SELECT juego_id, COUNT(*) as reviews_en_bd 
            FROM Hist_Steam_Reviews 
            GROUP BY juego_id
        ) r ON j.juego_id = r.juego_id
        WHERE j.id_steam IS NOT NULL 
          AND j.recommendations_count >= ? 
          AND j.steam_price_final IS NOT NULL
          -- Solo juegos que aun necesitan mas reviews
          AND COALESCE(r.reviews_en_bd, 0) < ?
        ORDER BY j.recommendations_count DESC
        --LIMIT ?
    """
    juegos = pd.read_sql_query(
        query, conn_principal, 
        #params=(min_reviews_juego, max_reviews_por_juego, limite_juegos), no usar limte que causaba problemas
        params=(min_reviews_juego, max_reviews_por_juego)
    )
    conn_principal.close()

    if juegos.empty:
        print("No hay juegos pendientes de procesar.")
        return

    total_juegos = len(juegos)
    print(f"Juegos pendientes: {total_juegos}")
    omitidos = 0

    for i, (_, juego) in enumerate(juegos.iterrows(), 1):
        appid = juego['id_steam']
        juego_id = juego['juego_id']
        reviews_en_bd = juego['reviews_en_bd']
        reviews_objetivo = min(max_reviews_por_juego, juego['recommendations_count'])
        reviews_restantes = reviews_objetivo - reviews_en_bd

        if reviews_restantes <= 0:
            omitidos += 1
            continue

        reviews_acumuladas = 0
        cursor = '*'
        lotes_sin_nuevas = 0  # Contador de lotes consecutivos sin insertar nada

        print(f"\n[{i}/{total_juegos}] {juego['titulo']} (AppID: {appid})")
        print(f"  En BD: {reviews_en_bd} | Objetivo: {reviews_objetivo} | Faltan: {reviews_restantes}")

        conn = sqlite3.connect(DB_PATH)
        #parametros para no batallar al consultar la base
        conn.execute("PRAGMA journal_mode=WAL")
        conn.execute("PRAGMA synchronous=NORMAL") 
        db_cursor_conn = conn.cursor()

        try:
            while reviews_acumuladas < reviews_restantes:
                cursor_encoded = urllib.parse.quote(cursor)
                url = (
                    f"https://store.steampowered.com/appreviews/{appid}"
                    f"?json=1&filter=recent&language=latam&purchase_type=all"#usar recent y latam que sin usando all y spanish nos trae menos reseñas
                    f"&cursor={cursor_encoded}&num_per_page=100"
                )

                try:
                    res = requests.get(url, timeout=15)
                    if res.status_code != 200:
                        print(f"  > Error HTTP {res.status_code}. Saltando juego.")
                        break

                    data = res.json()
                    if not data.get('success') or not data.get('reviews'):
                        print("  > Sin más reseñas disponibles en Steam.")
                        break

                    batch = data['reviews']
                    insert_data = [
                        (
                            str(r['recommendationid']),
                            juego_id,
                            r['review'],
                            1 if r['voted_up'] else 0,
                            r['votes_up'],
                            r['votes_funny'],
                            float(r['weighted_vote_score']),
                            r['author']['playtime_at_review'],
                            r['author']['playtime_forever'],
                            r['timestamp_created'],
                            r['author']['num_reviews'],
                            r['author']['num_games_owned'],
                            1 if r['received_for_free'] else 0,
                            1 if r['written_during_early_access'] else 0
                        )
                        for r in batch
                    ]

                    antes = db_cursor_conn.execute(
                        "SELECT COUNT(*) FROM Hist_Steam_Reviews WHERE juego_id = ?", (juego_id,)
                    ).fetchone()[0]

                    db_cursor_conn.executemany(
                        "INSERT OR IGNORE INTO Hist_Steam_Reviews VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
                        insert_data
                    )
                    conn.commit()

                    despues = db_cursor_conn.execute(
                        "SELECT COUNT(*) FROM Hist_Steam_Reviews WHERE juego_id = ?", (juego_id,)
                    ).fetchone()[0]

                    nuevas = despues - antes

                    reviews_acumuladas += len(batch)

                    print(
                        f"  > Lote: {len(batch)} obtenidas | {nuevas} nuevas "
                        f"(Total sesión: {reviews_acumuladas}/{reviews_restantes})"
                    )

                    # STOP EARLY: si Steam ya no tiene nada nuevo para nosotros
                    #para laprimera version duncionaba pero para rellenar la base con mas reviews solo causaba problemas
                    # if nuevas == 0:
                    #     lotes_sin_nuevas += 1
                    #     if lotes_sin_nuevas >= 2:  # 2 lotes consecutivos vacíos → parar
                    #         print("  > 2 lotes sin datos nuevos. Juego completo, siguiente.")
                    #         break
                    # else:
                    #     lotes_sin_nuevas = 0  # Resetear si volvieron a aparecer nuevas

                    nuevo_cursor = data.get('cursor')
                    # Steam a veces repite el cursor al llegar al final
                    if not nuevo_cursor or nuevo_cursor == cursor:
                        print("  > Cursor sin cambios. Fin de paginación.")
                        break
                    cursor = nuevo_cursor

                    if len(batch) < 100:
                        print("  > Último lote parcial. Fin de reseñas.")
                        break

                    time.sleep(0.5)

                except requests.exceptions.Timeout:
                    print("  > Timeout. Reintentando en 5s...")
                    time.sleep(5)
                    continue
                except Exception as e:
                    print(f"  > Error inesperado: {e}")
                    break

        finally:
            conn.close() #cerrar pase lo que pase

    if omitidos:
        print(f"\nResumen: {omitidos} juegos omitidos (ya tenían suficientes reviews en BD).")

In [10]:
descargar_reviews_steam(max_reviews_por_juego=1000, min_reviews_juego=100, limite_juegos=10000)

Juegos pendientes: 5520

[1/5520] Counter-Strike 2 (AppID: 730)
  En BD: 0 | Objetivo: 1000 | Faltan: 1000
  > Lote: 100 obtenidas | 100 nuevas (Total sesión: 100/1000)
  > Lote: 100 obtenidas | 100 nuevas (Total sesión: 200/1000)
  > Lote: 100 obtenidas | 100 nuevas (Total sesión: 300/1000)
  > Lote: 100 obtenidas | 100 nuevas (Total sesión: 400/1000)
  > Lote: 100 obtenidas | 100 nuevas (Total sesión: 500/1000)
  > Lote: 100 obtenidas | 100 nuevas (Total sesión: 600/1000)
  > Lote: 100 obtenidas | 100 nuevas (Total sesión: 700/1000)
  > Lote: 100 obtenidas | 100 nuevas (Total sesión: 800/1000)
  > Lote: 100 obtenidas | 100 nuevas (Total sesión: 900/1000)
  > Lote: 100 obtenidas | 100 nuevas (Total sesión: 1000/1000)

[2/5520] PUBG: Battlegrounds (AppID: 578080)
  En BD: 0 | Objetivo: 1000 | Faltan: 1000
  > Lote: 100 obtenidas | 100 nuevas (Total sesión: 100/1000)
  > Lote: 100 obtenidas | 100 nuevas (Total sesión: 200/1000)
  > Lote: 100 obtenidas | 100 nuevas (Total sesión: 300/100

In [ ]:
def resumen_final_reviews(cantidad_top=10):
    conn = sqlite3.connect(DB_PATH)
    query = """
    SELECT 
        j.titulo,
        COUNT(r.resena_id) AS total_resenas_descargadas,
        ROUND(AVG(r.recomendado) * 100, 2) || '%' AS recomendado,
        ROUND(AVG(r.minutos_totales / 60.0), 1) AS promedio_horas_jugadas
    FROM Hist_Steam_Reviews r
    JOIN CAT_Juego j ON r.juego_id = j.juego_id
    GROUP BY j.juego_id
    ORDER BY total_resenas_descargadas DESC
    LIMIT ?;
    """
    df = pd.read_sql_query(query, conn, params=(cantidad_top,))
    conn.close()
    
    display(df)


In [ ]:
resumen_final_reviews(50)


,titulo,total_resenas_descargadas,recomendado,promedio_horas_jugadas
0,Resident Evil Requiem,4430,98.44%,32.5
1,Counter-Strike 2,3259,84.23%,421.4
2,Crimson Desert,2100,91.71%,58.2
3,Left 4 Dead 2,1979,97.98%,203.9
4,Geometry Dash,1786,94.18%,135.6
5,Resident Evil 2,967,98.76%,23.8
6,Resident Evil 4,966,99.28%,44.0
7,Resident Evil 3,955,94.03%,12.7
8,Terraria,891,96.97%,154.3
9,Project Zomboid,876,95.78%,179.4


In [11]:
def inspeccionar_resenas(busqueda, es_id=False):
    conn = sqlite3.connect(DB_PATH)
    
    if es_id:
        filtro = "j.juego_id = ?"
        parametro = busqueda
    else:
        filtro = "j.titulo LIKE ?"
        parametro = f"%{busqueda}%"

    query = f"""
    SELECT 
        j.juego_id,
        j.titulo,
        r.recomendado,
        r.minutos_totales / 60 AS horas_jugadas,
        r.votos_utiles,
        r.resena_texto
    FROM Hist_Steam_Reviews r
    JOIN CAT_Juego j ON r.juego_id = j.juego_id
    WHERE {filtro}
    --ORDER BY r.votos_utiles DESC
    --LIMIT 10;
    """
    
    df = pd.read_sql_query(query, conn, params=(parametro,))
    conn.close()
    
    if df.empty:
        print(f"No se encontraron reseñas para: {busqueda}")
    else:
        display(df)

In [13]:
#Ara que tratar de alguna forma las malas palabras de los jeugos ya que se ve que son creativos con las opiniones
inspeccionar_resenas("Suicide Squad: Kill the Justice League")


,juego_id,titulo,recomendado,horas_jugadas,votos_utiles,resena_texto
0,2639,Suicide Squad: Kill the Justice League,1,22,0,excelente
1,2639,Suicide Squad: Kill the Justice League,0,32,0,muy repetitivo
2,2639,Suicide Squad: Kill the Justice League,0,20,0,"Este juego tiene un buen concepto, pero lament..."
3,2639,Suicide Squad: Kill the Justice League,1,28,0,ufff cine a mi poarecer xD
4,2639,Suicide Squad: Kill the Justice League,1,10,0,Dentro de todo y por el precio si me parece un...
...,...,...,...,...,...,...
357,2639,Suicide Squad: Kill the Justice League,1,45,0,me encanta este juego. lo disfrutaría mas si n...
358,2639,Suicide Squad: Kill the Justice League,1,61,3,Pues si no sos como el procentaje de gente que...
359,2639,Suicide Squad: Kill the Justice League,1,74,0,Goty
360,2639,Suicide Squad: Kill the Justice League,0,24,98,Mal rendimiento al desplazarse por la ciudad (...


In [ ]:
inspeccionar_resenas("pubg")

,juego_id,titulo,recomendado,horas_jugadas,votos_utiles,resena_texto
0,226,PUBG: Battlegrounds,1,320,0,"Excelente juego, mundo abierto y realista con ..."
1,226,PUBG: Battlegrounds,1,1087,0,"Me gusta el juego, siempre estan cambiando cos..."
2,226,PUBG: Battlegrounds,0,300,3,"Paso de jugar, es imposible con la cantidad de..."
3,226,PUBG: Battlegrounds,0,68,1,Imagina ser un muerto de hambre y sacar a la c...
4,226,PUBG: Battlegrounds,1,682,2,Gracias por poner la posibilidad de meter a un...
...,...,...,...,...,...,...
251,226,PUBG: Battlegrounds,1,36,0,L0 R3comiendo un JuegaZo0 lOs p3n3s tienen bue...
252,226,PUBG: Battlegrounds,0,160,0,eafresgvdrv
253,226,PUBG: Battlegrounds,1,397,0,...
254,226,PUBG: Battlegrounds,1,374,0,el mejoir juego de tiros de la historia si mej...
